# UMUD R22 A — nowy model i pięć foldów

Punkt odniesienia: R21 **0,38563**, miejsce 15, lider **0,28263** (informacja użytkownika).

ConvNeXt Tiny z ImageNet + pełna rozdzielczość + nadzór kierunku na liniach GT. Pięć niezależnych treningów po 24 epoki, potem ensemble. Wymaga GPU, Internetu do pobrania publicznych wag i ukończonego cache R17. To może być kilka sesji GPU; zacznij od A100/L4, jeśli masz dostęp, albo T4.

**Wznowienie:** uruchom ponownie wszystkie komórki, zachowaj ten sam katalog RUN. Ostatnia niedokończona epoka zostanie powtórzona. Nie uruchamiaj dwóch sesji zapisujących ten sam fold. OOF to kontrola segmentacji na grupach wizualnych, nie nowe niezależne etykiety PA/FL. Wyniki OOF nie wybierają epoki ani progu. Przygotowany kod nie oznacza ukończonego treningu.

## 1. Drive i kod

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import base64,hashlib,importlib.util,json,subprocess,sys,zlib
PROJECT=Path('/content/drive/MyDrive/UMUD_Challenge')
RUN=PROJECT/'outputs/R22_v1_0_0'
R17=PROJECT/'outputs/R17_v1_0_0'
RUN.mkdir(parents=True,exist_ok=True)
SOURCE=RUN/'source';SOURCE.mkdir(exist_ok=True)
PACKAGE_SHA='9fe008a84b0a26f3bd0bc07f4ddc60377fb0ce89afb4d79805c91c10684a72f4'
raw=zlib.decompress(base64.b64decode('...'))
assert hashlib.sha256(raw).hexdigest()==PACKAGE_SHA
if (RUN/'RUN_PROTOCOL.json').exists():
    assert json.loads((RUN/'RUN_PROTOCOL.json').read_text())['package_sha256']==PACKAGE_SHA, 'Kod zmieniony: użyj nowego katalogu RUN'
for relative,content in json.loads(raw).items():
    path=SOURCE/relative;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(content)
for module,pip_name in [('cv2','opencv-python-headless'),('skimage','scikit-image'),('timm','timm'),('pandas','pandas')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pip_name])
sys.path.insert(0,str(SOURCE))
import cv2,numpy as np,pandas as pd,torch,torchvision,skimage,timm
from umud_core import sha256,atomic_json
from r22_model import CONFIG,VERSION
ASSETS=SOURCE/'assets'
assert torch.cuda.is_available(), 'W Colab wybierz: Środowisko wykonawcze → Zmień typ → GPU; potem uruchom wszystkie komórki.'
DEVICE='cuda'
VERSIONS={'torch':str(torch.__version__),'torchvision':str(torchvision.__version__),'numpy':np.__version__,
          'opencv':cv2.__version__,'skimage':skimage.__version__,'timm':timm.__version__}
print(VERSION,'GPU:',torch.cuda.get_device_name(0),VERSIONS)


Mounted at /content/drive
R22_v1.0.0 GPU: Tesla T4 {'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.1.3', 'opencv': '5.0.0', 'skimage': '0.25.2', 'timm': '1.0.29'}


## 2. Zamrożony protokół

In [2]:
# Ustawienia operacyjne. W razie rozłączenia: ponownie Uruchom wszystkie.
FOLDS=[0,1,2,3,4]
SESSION_HOURS=8
# Konfiguracja naukowa jest zamrożona w CONFIG; nie stroimy jej na OSF ani LB.
from r22_data import prepare_cache,stage_cache,digest,Dataset
from r22_model import ensure_pretrained,initialize,loss_fn,seed_all
from r22_train import train_all
import gc,time
samples=pd.read_csv(ASSETS/'samples.csv')
assert len(samples)==2046 and samples.sample_id.is_unique
assert samples.groupby('group_id').fold.nunique().max()==1
assert set(FOLDS).issubset({0,1,2,3,4}) and len(FOLDS)==len(set(FOLDS)) and FOLDS
cache_config=json.loads((R17/'cache_config.json').read_text())
assert cache_config['identity']=='1e73f806e5d9118b2f95d2384196b88f39031fdb4b0fab5df242a97cbff31771', 'Inny cache R17'
protocol={'version':VERSION,'package_sha256':PACKAGE_SHA,'config':CONFIG,'versions':VERSIONS,
          'samples_sha256':sha256(ASSETS/'samples.csv'),'r17_cache_identity':cache_config['identity'],
          'anchor_sha256':sha256(ASSETS/'anchor_038563.csv'),
          'training_policy':'24_FIXED_EPOCHS_IMAGE_NET_INIT_NO_HELDOUT_SELECTION',
          'sampler':'.5 uniform images + .5 uniform visual groups within training folds only'}
protocol_path=RUN/'RUN_PROTOCOL.json'
if protocol_path.exists():
    assert json.loads(protocol_path.read_text())==protocol, 'Konfiguracja lub biblioteki zmienione. Przywróć wersje z RUN_PROTOCOL.json; dla nowego eksperymentu użyj nowego RUN.'
else:atomic_json(protocol_path,protocol)
print(samples.groupby('fold').agg(images=('sample_id','size'),groups=('group_id','nunique')))
print('Każdy fold startuje z ImageNet, bez wag R17 refit. Model: 31,1 mln parametrów.')


      images  groups
fold                
0        370       1
1        300       1
2        463     231
3        459     229
4        454     240
Każdy fold startuje z ImageNet, bez wag R17 refit. Model: 31,1 mln parametrów.


## 3. Cache i publiczne wagi

In [3]:
pretrained=ensure_pretrained(RUN/'pretrained')
cache_index=prepare_cache(samples,R17,RUN/'cache',CONFIG['side'])
CACHE=stage_cache(samples,RUN/'cache',Path('/content/UMUD_R22_cache'))
print('Pokrycie kierunku GT:',float(cache_index.direction_pixels.sum()/max(1,cache_index.line_pixels.sum())))
print('Cache gotowy; wagi, checkpointy i raporty są w:',RUN)


R22 cache 100 / 2046
R22 cache 200 / 2046
R22 cache 300 / 2046
R22 cache 400 / 2046
R22 cache 500 / 2046
R22 cache 600 / 2046
R22 cache 700 / 2046
R22 cache 800 / 2046
R22 cache 900 / 2046
R22 cache 1000 / 2046
R22 cache 1100 / 2046
R22 cache 1200 / 2046
R22 cache 1300 / 2046
R22 cache 1400 / 2046
R22 cache 1500 / 2046
R22 cache 1600 / 2046
R22 cache 1700 / 2046
R22 cache 1800 / 2046
R22 cache 1900 / 2046
R22 cache 2000 / 2046
R22 cache 2046 / 2046
local cache 250 / 2046
local cache 500 / 2046
local cache 750 / 2046
local cache 1000 / 2046
local cache 1250 / 2046
local cache 1500 / 2046
local cache 1750 / 2046
local cache 2000 / 2046
Pokrycie kierunku GT: 0.9998168347769704
Cache gotowy; wagi, checkpointy i raporty są w: /content/drive/MyDrive/UMUD_Challenge/outputs/R22_v1_0_0


## 4. Test GPU

In [4]:
# Rzeczywisty test pełnej rozdzielczości na przydzielonym GPU, przed długim treningiem.
seed_all(CONFIG['seed']);torch.cuda.reset_peak_memory_stats()
probe=initialize(pretrained).to(DEVICE).train()
batch=next(iter(torch.utils.data.DataLoader(Dataset(samples.head(1),CACHE),batch_size=1)))
x,y,v,line,d,w=[t.to(DEVICE) for t in batch]
started=time.time()
with torch.amp.autocast('cuda',dtype=torch.float16):
    output=probe(x);loss,seg,axial=loss_fn(output,y,v,line,d,w,10.,CONFIG['orientation_weight'])
assert torch.isfinite(loss)
loss.backward();torch.cuda.synchronize()
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in probe.parameters())
preflight={'version':VERSION,'shape':list(output.shape),'forward_backward_seconds':time.time()-started,
           'peak_allocated_GiB':torch.cuda.max_memory_allocated()/2**30,'gpu':torch.cuda.get_device_name(0),
           'parameters':sum(p.numel() for p in probe.parameters()),'pretrained_sha256':sha256(pretrained)}
atomic_json(RUN/'PREFLIGHT.json',preflight);print(preflight)
del probe,batch,x,y,v,line,d,w,output,loss,seg,axial;gc.collect();torch.cuda.empty_cache()
print('Test modelu zakończony. Czas pełnego treningu oszacuj z seconds w pierwszej zapisanej epoce.')
print('5 foldów × 24 epoki, rozmiar treningu różni się między foldami. Nie obiecujemy czasu jednej sesji.')


{'version': 'R22_v1.0.0', 'shape': [1, 3, 768, 768], 'forward_backward_seconds': 12.333727598190308, 'peak_allocated_GiB': 2.1896066665649414, 'gpu': 'Tesla T4', 'parameters': 31056347, 'pretrained_sha256': '00ce60f960437b258b304cc6a8b35feab2aca32c651fd1d18fca63432a48cd11'}
Test modelu zakończony. Czas pełnego treningu oszacuj z seconds w pierwszej zapisanej epoce.
5 foldów × 24 epoki, rozmiar treningu różni się między foldami. Nie obiecujemy czasu jednej sesji.


## 5. Trening z wznawianiem

In [5]:
status=train_all(samples,CACHE,RUN,pretrained,CONFIG,protocol,FOLDS,DEVICE,SESSION_HOURS)
print(json.dumps(status,indent=2,ensure_ascii=False))
if not status['complete']:
    print('To jeszcze nie koniec treningu. Połącz GPU i uruchom ponownie wszystkie komórki A. Wznowienie od ostatniej pełnej epoki.')
else:
    display(pd.read_csv(RUN/'OOF_BY_FOLD.csv'))
    print('PIĘĆ FOLDÓW GOTOWYCH. Uruchom notebook B.')


{'fold': 0, 'epoch': 3, 'step': 50, 'steps': 1676, 'mean_loss': 0.4258933359384537, 'last_safe_epoch': 2, 'time': 1789813396.0059865}
{'fold': 0, 'epoch': 3, 'step': 100, 'steps': 1676, 'mean_loss': 0.3999183714389801, 'last_safe_epoch': 2, 'time': 1789813407.8142738}
{'fold': 0, 'epoch': 3, 'step': 150, 'steps': 1676, 'mean_loss': 0.39865928490956626, 'last_safe_epoch': 2, 'time': 1789813419.577218}
{'fold': 0, 'epoch': 3, 'step': 200, 'steps': 1676, 'mean_loss': 0.3962312875688076, 'last_safe_epoch': 2, 'time': 1789813431.3044784}
{'fold': 0, 'epoch': 3, 'step': 250, 'steps': 1676, 'mean_loss': 0.3952355705499649, 'last_safe_epoch': 2, 'time': 1789813443.3509579}
{'fold': 0, 'epoch': 3, 'step': 300, 'steps': 1676, 'mean_loss': 0.39284718801577884, 'last_safe_epoch': 2, 'time': 1789813455.5005944}
{'fold': 0, 'epoch': 3, 'step': 350, 'steps': 1676, 'mean_loss': 0.3935356544596808, 'last_safe_epoch': 2, 'time': 1789813467.6815147}
{'fold': 0, 'epoch': 3, 'step': 400, 'steps': 1676, 'me

RuntimeError: Nonfinite gradient

## 6. Raport

In [ ]:
import zipfile
training_zip=RUN/'UMUD_R22_TRENING_RAPORT.zip'
with zipfile.ZipFile(training_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ('RUN_PROTOCOL.json','PREFLIGHT.json','TRAINING_COMPLETE.json','OOF.csv','OOF_BY_FOLD.csv'):
        p=RUN/name
        if p.exists():z.write(p,name)
    for pattern in ('fold_*/history.csv','fold_*/heartbeat.json','fold_*/DONE.json'):
        for p in RUN.glob(pattern):z.write(p,str(p.relative_to(RUN)))
print('Raport:',training_zip)
